# Step 4 — Case studies, hidden Markov model, monitor

**RSM434 Industry Project 3 · Option strategies through market regimes**

The metrics in Step 3 are averages over every month that followed a label. These windows are the paths. Three are the ones in the brief. The fourth is the financial crisis, from the Lehman filing on 15 Sep 2008 through the S&P low on 9 Mar 2009. It is the deep bear the other three are not, and it is the window where the spread across strategies is widest.

A warning is the last month-end strictly before the window. Month-ends inside the window are concurrent. The eve reading applies the same rules on the last VIX session before the start, so a move between the month-end and the window is visible. It is labeled as that, not as an official month-end.

The notebook then fits a 3-state hidden Markov model, writes the compiled workbook, the regime monitor and `output/deck_notes.md`.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

import ip3

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 80)

daily, readings, paired = ip3.load_processed()
month_ends = ip3.month_end_calendar(daily)
check = ip3.month_end_reading_check(daily, month_ends, readings)
if not check.empty:
    raise RuntimeError("a month-end reading disagrees with compute_signals")
print(f"Reading check passed on {len(readings)} month-ends.")

stats, wealth, context = ip3.run_episodes(daily, month_ends, readings)
context_cols = [
    "episode", "prior_date", "prior_vix", "prior_vix_median", "prior_vol_regime", "prior_credit",
    "eve_date", "eve_is_month_end", "eve_vix", "eve_vix_change_1m", "eve_vol_regime", "eve_credit",
    "vix_max", "vix_max_date", "during",
]
display(context[context_cols])

returns = stats.pivot(index="asset", columns="episode", values="total_return").reindex(ip3.ASSETS)
returns = returns[["Volmageddon", "COVID crash", "2022 bear market", "Financial crisis"]]
print("Total return inside the window (%)")
display((returns * 100).round(1))
print("Peak-to-trough drawdown inside the window (%)")
drawdown = stats.pivot(index="asset", columns="episode", values="max_drawdown").reindex(returns.index)[returns.columns]
display((drawdown * 100).round(1))


## Paths

Each line is rebased to 100 at that series' first print in the window. The S&P is black. Equity-collateral books (the buy-writes, the collar, PPUT) track it when the put or the call does not change the shape much. CNDR and BFLY should not, because they hold T-bills and sell a defined range.


In [ ]:
order = ["volmageddon", "covid", "bear_2022", "gfc"]
names = {item["id"]: item["name"] for item in ip3.EPISODES}
class _DayMonth(mdates.DateFormatter):
    def __init__(self):
        super().__init__("%d %b")
    def __call__(self, x, pos=0):
        stamp = mdates.num2date(x).replace(tzinfo=None)
        return f"{stamp.day} {stamp:%b}"

fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.5), sharey=False)
for ax, key in zip(axes.ravel(), order):
    path = wealth[key]
    for asset in ip3.ASSETS:
        kw = {"color": "black", "lw": 1.8, "zorder": 3} if asset == "SPX_TR" else {"lw": 1.1}
        ax.plot(path.index, path[asset], label=asset, **kw)
    ax.set_title(names[key])
    ax.axhline(100, color="grey", lw=0.6)
    if key in ("bear_2022", "gfc"):
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    else:
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
        ax.xaxis.set_major_formatter(_DayMonth())
    ax.tick_params(axis="x", labelrotation=30)
    ax.margins(y=0.12)
axes[0, 0].legend(ncol=3, fontsize=7, loc="lower left")
fig.suptitle("Wealth, rebased to 100 at the first print in each window", y=1.01)
fig.tight_layout()
ip3.OUTPUT.mkdir(parents=True, exist_ok=True)
fig.savefig(ip3.OUTPUT / "episode_paths.png", dpi=150, bbox_inches="tight")
plt.show()


## Could the reading have warned?

Volmageddon opens with VIX near 11. The December 2017 label is High only because 11 sat just above a trailing median near 11, and credit was tightening. The eve of the window flips the direction to Rising while VIX is still about 12. Neither reading is a stress call. The loss is shared across the books, within about six percentage points, and the 5% put cushions only part of a two-week drop.

COVID is the opposite shape. The 31 Jan 2020 month-end is already High/Rising, with VIX above its median, while credit is still tightening. By 18 Feb VIX has fallen and the same rules read High/Falling. The warning was there at month-end and had faded the day before the crash window. Inside the window the buy-writes and the put-write fall with the S&P. CLL, PPUT, CNDR and BFLY lose much less. That is the crash the outright put and the wings are built for.

The 2022 window starts on the first session of the year, so the eve reading is the 31 Dec 2021 month-end: Low/Falling, because VIX was below its median, with credit widening. VIX peaks in the mid-30s. PPUT falls almost with the equity. A one-month 5% put, rolled through a grind, still leaves the strategy long the S&P. CNDR and BFLY lose about 5%. Credit was the signal that had flipped. The volatility grid had not.

The crisis window is the one added here. On 29 Aug 2008 the rules say Low/Falling, VIX below a median that had already risen, and credit widening: the same combination as December 2021. On 12 Sep, still before the window, the same rules read High/Rising and credit is still widening. A desk that only classifies at month-end carries the calm label into the week of 15 Sep. From there the S&P falls about 42%. CNDR loses about 2% and the put-write about 31%, on the same T-bill collateral. The condor's wings are the difference. CLL and PPUT cut the loss versus the index and still lose about a fifth, because both hold the equity.


## Hidden Markov model

Three Gaussian states, on the same month-end ingredients the rules use: log VIX, the one-month VIX change, and the three-month credit change. The features are standardized with the full-sample mean and standard deviation, so this is a comparison with the rules, not a signal that could have been built in real time. Ten random restarts, seed 434 plus the restart number. States are named Calm, Transition and Stress by ascending average VIX, so the names do not depend on the optimizer's index.

`hmm_state` is the Viterbi path. It uses the whole sample. The `p_*` columns are filtered probabilities: the probability of the state given only the months up to that one. Those are the probabilities used in the episode lines. On the first month of the sample the filtered probability is the estimated initial distribution. August 2008 is that month, and the fit puts the start in Stress, so the August probability is not a warning.


In [ ]:
hmm = ip3.fit_hmm(readings)
print(f"In-sample log-likelihood {hmm['score']:.1f}. Converged: {hmm['converged']}. Seed {hmm['seed']}.")
print("Average VIX in each Viterbi state")
display(hmm["frame"].groupby("hmm_state")["vix"].mean().round(1).rename("mean VIX"))
print("Months in each pair of labels")
display(hmm["counts"])
print("Share of each rule regime (%)")
display((hmm["share"] * 100).round(1))
print("Transition matrix, rows from, columns to, order", list(ip3.HMM_STATES))
display(pd.DataFrame(hmm["model"].transmat_, index=ip3.HMM_STATES, columns=ip3.HMM_STATES).round(3))

context = ip3.attach_hmm_to_episodes(context, hmm["frame"])
print("Filtered probability on the last month-end before each window")
display(context[["episode", "prior_date", "prior_vol_regime", "prior_hmm_state", "prior_p_calm", "prior_p_transition", "prior_p_stress"]].round(3))

frame = hmm["frame"]
fig, axes = plt.subplots(2, 1, figsize=(12.5, 6.2), sharex=True)
axes[0].plot(frame.index, frame["vix"], color="black", lw=0.8)
axes[0].set_ylabel("VIX")
axes[0].set_title("VIX and filtered probability of the HMM stress state")
axes[1].plot(frame.index, frame["p_Stress"], color="#9b2c2c", lw=1.0)
axes[1].set_ylabel("P(Stress)")
axes[1].set_ylim(-0.02, 1.02)
for key, path in wealth.items():
    for ax in axes:
        ax.axvspan(path.index[0], path.index[-1], color="#f4c7c3", lw=0)
fig.tight_layout()
fig.savefig(ip3.OUTPUT / "hmm_stress_probability.png", dpi=150)
plt.show()


The states describe the level of VIX: roughly 15, 21 and 34. They are sticky. They do not reproduce the four rule labels. Only a minority of High/Rising months fall in HMM Stress, because High on the rule means "above the last year's median," and a VIX of 11 can clear that bar. The HMM calls that month Calm. Filtered P(Stress) is about zero before Volmageddon, before COVID and before the 2022 window. On 31 Jan 2020 the rule is already High/Rising and the HMM is still Calm. The level model is a check on the word High. It does not turn earlier than the rules.

The shaded bands on the chart are the four case-study windows.


## Workbook, monitor, deck notes

`output/ip3_compiled.xlsx` is the compiled data: the panel, the metrics, the half-sample check, the four windows, the HMM path and a snapshot of the monitor. `output/regime_monitor.xlsx` applies the same rules to the latest saved VIX close and shows how the strategies behaved after that volatility regime. `output/deck_notes.md` is the paragraph version of the tables, for the slides. The `deck_*.png` files in `output/` are sized for the slides.


In [ ]:
reading = ip3.current_reading(daily, month_ends=month_ends)
last_month_end = readings.iloc[-1]
notes = ip3.write_study_outputs(
    paired, readings,
    ip3.performance_table(paired),
    ip3.performance_table(paired, "vol_regime"),
    ip3.performance_table(paired, "regime_credit"),
    ip3.performance_table(paired, "regime_slope"),
    ip3.stability_table(paired),
    stats, context, hmm, reading, last_month_end,
)
as_of = pd.Timestamp(reading["as_of"])
print(f"As of {as_of.day} {as_of:%b %Y}: {reading['vol_regime']}, credit {reading['credit_signal']}.")
print("Wrote output/ip3_compiled.xlsx, output/regime_monitor.xlsx, output/deck_notes.md")
print(notes)
